# Day 5 practice — Testing without a server, without a database

**Read first:** [09_theory_testing_apis.md](09_theory_testing_apis.md)

You'll write real tests, run **real pytest** as a subprocess, read a genuine failure message, and
swap a database for cardboard in one line.

Everything here runs with no Postgres and no uvicorn.

In [ ]:
import json
import subprocess
import sys
import textwrap
from pathlib import Path

from fastapi import Depends, FastAPI, HTTPException, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel

SANDBOX = Path("sandbox")          # git-ignored scratch space for real test files
SANDBOX.mkdir(exist_ok=True)

def write(name: str, code: str) -> Path:
    path = SANDBOX / name
    path.write_text(textwrap.dedent(code).lstrip(), encoding="utf-8")
    return path

def run_pytest(*args: str) -> None:
    """Run the real pytest binary and print exactly what you'd see in a terminal."""
    result = subprocess.run(
        [sys.executable, "-m", "pytest", *args],
        cwd=SANDBOX, capture_output=True, text=True,
    )
    print(result.stdout[-3000:])
    if result.stderr.strip():
        print("STDERR:", result.stderr[-800:])

print("ready. sandbox:", SANDBOX.resolve())

---

## Part 1 — The app under test

A small API with a database dependency. Note it **asks** for its connection — that one design
decision is what makes everything below possible.

In [ ]:
app_code = '''
    from fastapi import Depends, FastAPI, HTTPException

    app = FastAPI()

    def get_conn():
        # Production: a real database connection. Tests will replace this.
        raise RuntimeError("no database configured")   # deliberately explosive

    @app.get("/health")
    def health():
        return {"status": "ok"}

    @app.get("/cities")
    def list_cities(limit: int = 10, conn = Depends(get_conn)):
        return conn.query("SELECT city FROM cities")[:limit]

    @app.get("/cities/{name}")
    def read_city(name: str, conn = Depends(get_conn)):
        rows = [r for r in conn.query("SELECT city FROM cities") if r["city"] == name]
        if not rows:
            raise HTTPException(404, detail=f"City {name!r} not found")
        return rows[0]

    @app.get("/users/{user_id}")
    def read_user(user_id: int):
        return {"id": user_id, "email": "a@b.c", "password_hash": "$2b$LEAK"}
'''
write("app_under_test.py", app_code)
print("app_under_test.py written")

---

## Part 2 — `TestClient`: HTTP with no HTTP

In [ ]:
sys.path.insert(0, str(SANDBOX.resolve()))
from app_under_test import app, get_conn      # noqa: E402

client = TestClient(app)

r = client.get("/health")
print("status  :", r.status_code)
print("json    :", r.json())
print("headers :", dict(list(r.headers.items())[:3]))
print()
print("No server started. No port opened. Same code path as production.")

### 🔮 Predict

`/cities` depends on `get_conn`, which raises `RuntimeError`. What status code comes back, and is
that the right one?

In [ ]:
r = TestClient(app, raise_server_exceptions=False).get("/cities")
print("GET /cities ->", r.status_code, r.text[:80])
print()
print("500: our dependency exploded. That IS our fault, so 5xx is correct.")
print("Now let's give it something that works instead.")

---

## Part 3 — `dependency_overrides`: the cardboard prep station

In [ ]:
class FakeConnection:
    def __init__(self, rows):
        self.rows = rows
        self.queries = []

    def query(self, sql):
        self.queries.append(sql)
        return self.rows

FAKE_ROWS = [{"city": "Utrecht"}, {"city": "Amsterdam"}, {"city": "Rotterdam"}]

def fake_conn():
    yield FakeConnection(FAKE_ROWS)

app.dependency_overrides[get_conn] = fake_conn       # <- the entire trick

r = client.get("/cities")
print("GET /cities        ->", r.status_code, r.json())
r = client.get("/cities/Utrecht")
print("GET /cities/Utrecht->", r.status_code, r.json())
r = client.get("/cities/Atlantis")
print("GET /cities/Atlantis->", r.status_code, r.json())

app.dependency_overrides.clear()                     # ALWAYS clean up
print()
print("after clear(), /cities is broken again:",
      TestClient(app, raise_server_exceptions=False).get("/cities").status_code)

**No handler was edited.** One dictionary entry redirected every `Depends(get_conn)` in the app.

That only worked because the handler *asks* for its connection. Had it called `create_engine(URL)`
in its own body there would be no seam, and your only lever would be monkey-patching module globals
— fragile, and it breaks whenever someone moves an import.

> 🎯 **"Ask, don't build" isn't style advice. It's what creates the seam tests need.**

### Why the cleanup matters

`dependency_overrides` lives on the shared `app` object. Forget to clear it and it leaks into
unrelated tests, producing the worst kind of failure: one that depends on test **order**. A fixture
with `yield` makes the cleanup automatic — that's what you'll write next.

---

## Part 4 — Real pytest, real output

Everything so far ran inline. Now write actual test files and run the real binary.

In [ ]:
write("conftest.py", '''
    import pytest
    from fastapi.testclient import TestClient

    from app_under_test import app, get_conn

    FAKE_ROWS = [{"city": "Utrecht"}, {"city": "Amsterdam"}, {"city": "Rotterdam"}]

    class FakeConnection:
        def query(self, sql):
            return FAKE_ROWS

    @pytest.fixture
    def client():
        # A TestClient whose database is made of cardboard.
        # The teardown after `yield` is what stops this override leaking
        # into other tests.
        def fake_conn():
            yield FakeConnection()

        app.dependency_overrides[get_conn] = fake_conn
        yield TestClient(app)
        app.dependency_overrides.clear()
''')

write("test_cities.py", '''
    import pytest


    def test_health(client):
        r = client.get("/health")
        assert r.status_code == 200
        assert r.json() == {"status": "ok"}


    def test_list_cities(client):
        r = client.get("/cities")
        assert r.status_code == 200
        assert len(r.json()) == 3


    def test_limit_is_respected(client):
        assert len(client.get("/cities", params={"limit": 2}).json()) == 2


    def test_unknown_city_is_404(client):
        r = client.get("/cities/Atlantis")
        assert r.status_code == 404
        assert "Atlantis" in r.json()["detail"]


    @pytest.mark.parametrize("limit", [0, -1, "abc"])
    def test_bad_limit_is_422(client, limit):
        assert client.get("/cities", params={"limit": limit}).status_code == 422
''')
run_pytest("-v")

### 🔮 Predict

`test_bad_limit_is_422` expects a `422` for `limit=0` and `limit=-1`. But the handler is
`def list_cities(limit: int = 10, ...)` — nothing declares a minimum.

Which of the three parametrised cases will actually fail, and why?

Look at the run above. `limit=0` and `limit=-1` **fail**, because `0` and `-1` are perfectly
valid integers — the handler never said otherwise. Only `"abc"` produces a `422`.

**The test was right and the code was wrong.** That's a test earning its keep. Fix the code:

In [ ]:
write("app_under_test.py", app_code.replace(
    "def list_cities(limit: int = 10, conn = Depends(get_conn)):",
    "def list_cities(limit: int = Query(default=10, ge=1, le=100), conn = Depends(get_conn)):"
).replace(
    "from fastapi import Depends, FastAPI, HTTPException",
    "from fastapi import Depends, FastAPI, HTTPException, Query"
))
run_pytest("-v")

Green. `Query(ge=1, le=100)` made `0` and `-1` invalid, and you wrote no `if` statement.

---

## Part 5 — Reading a failure properly

A failing test is only useful if you can read it. Let's write one that fails on purpose.

In [ ]:
write("test_failure_demo.py", '''
    def test_brittle_exact_equality(client):
        # This is the WRONG way to assert, and here's what it costs you.
        assert client.get("/cities").json() == [{"city": "Utrecht"}]
''')
run_pytest("test_failure_demo.py", "-v")

Read the output above. pytest shows you:

| Part | What it tells you |
|---|---|
| `test_failure_demo.py::test_brittle_exact_equality FAILED` | Which test |
| The `>` line | The exact assertion that failed |
| `E   assert [...] == [...]` | Both sides, expanded |
| `E     At index 1 diff:` | **Where** they diverge |
| `short test summary info` | A one-line recap at the bottom |

The bottom summary is the bit to read first when twenty tests fail.

### Why that test was badly written

It asserted the **exact payload**. Add a city — a completely correct change — and it goes red. A
suite that cries wolf is a suite people stop reading.

Assert the **contract** instead:

In [ ]:
write("test_failure_demo.py", '''
    def test_contract_not_payload(client):
        r = client.get("/cities")
        assert r.status_code == 200                       # 1. status
        body = r.json()
        assert isinstance(body, list) and body            # 2. shape
        assert all("city" in row for row in body)         # 3. every row has the key
        assert "Utrecht" in [row["city"] for row in body] # 4. the value we care about
''')
run_pytest("test_failure_demo.py", "-v")

---

## Part 6 — The security assertion nobody writes

Here is the one place where exact equality **is** correct: when the complete key set is the promise.

In [ ]:
write("test_no_leak.py", '''
    def test_user_response_leaks_a_password_hash(client):
        # This test SHOULD fail - the endpoint has no response_model.
        body = client.get("/users/42").json()
        assert set(body) == {"id", "email"}, f"leaked: {set(body) - {'id', 'email'}}"
''')
run_pytest("test_no_leak.py", "-v")

The failure message names the leaked field. Now fix the endpoint the Day 3 way — with a
`response_model` allow-list — and watch it go green with no change to the test.

In [ ]:
fixed_app = app_code.replace(
    "from fastapi import Depends, FastAPI, HTTPException",
    "from fastapi import Depends, FastAPI, HTTPException, Query\n    from pydantic import BaseModel"
).replace(
    "def list_cities(limit: int = 10, conn = Depends(get_conn)):",
    "def list_cities(limit: int = Query(default=10, ge=1, le=100), conn = Depends(get_conn)):"
).replace(
    '    @app.get("/users/{user_id}")',
    '    class UserOut(BaseModel):\n        id: int\n        email: str\n\n    @app.get("/users/{user_id}", response_model=UserOut)'
)
write("app_under_test.py", fixed_app)
run_pytest("test_no_leak.py", "-v")

Same test, same handler body, different result — because the response model dropped the
undeclared field on the way out.

> 🎯 **Every endpoint deserves two tests people usually skip: one failure case, and one that proves
> nothing leaks.**

---

## Part 7 — The pytest flags you'll actually use

In [ ]:
for label, args in [
    ("everything, quiet",        ("-q",)),
    ("one file",                 ("test_cities.py", "-q")),
    ("one test",                 ("test_cities.py::test_health", "-v")),
    ("by name substring",        ("-k", "404", "-v")),
    ("stop at first failure",    ("-x", "-q")),
]:
    print("=" * 70)
    print(label, "->", "pytest", " ".join(args))
    print("=" * 70)
    run_pytest(*args)

---

## Part 8 — Skipped is not passed

Integration tests skip when there's no database. That is polite and correct — and it means a green
run can prove nothing at all.

In [ ]:
write("test_integration_demo.py", '''
    import pytest

    def database_is_reachable() -> bool:
        return False          # pretend Docker is stopped

    @pytest.fixture(scope="module")
    def real_db():
        if not database_is_reachable():
            pytest.skip("No database reachable - start docker compose")
        yield "engine"

    def test_real_sql(real_db):
        assert real_db == "engine"
''')
run_pytest("-q", "-rs")          # -rs = report the REASON for each skip

Look at the summary line: **`N passed, 1 skipped`**. Not `N+1 passed`.

`-rs` printed *why* it skipped. Get into the habit of reading that line — the difference between
"my SQL is tested" and "my SQL has never once been executed" is invisible otherwise.

This is precisely why the project's `ci.yml` has a **separate job** with a Postgres service
container, and a step that fails the build if the integration tests skip.

---

## Exercises

### Exercise 1 — Cover an endpoint properly (⭐)

Write four tests for `GET /cities/{name}`: the happy path, a 404, that the response has a `city` key,
and that a name with a space (`Den Haag`) is handled. Put them in `sandbox/test_exercise1.py` and run
them.

In [ ]:
# Your code here - use write("test_exercise1.py", '''...''') then run_pytest("test_exercise1.py", "-v")


<details>
<summary>💡 Solution</summary>

```python
write("test_exercise1.py", '''
    def test_known_city(client):
        r = client.get("/cities/Utrecht")
        assert r.status_code == 200
        assert r.json()["city"] == "Utrecht"

    def test_unknown_city_is_404(client):
        r = client.get("/cities/Atlantis")
        assert r.status_code == 404
        assert "Atlantis" in r.json()["detail"]

    def test_response_has_city_key(client):
        assert "city" in client.get("/cities/Utrecht").json()

    def test_name_with_a_space(client):
        # The fake has no "Den Haag", so 404 is correct - the point is that the
        # space is URL-encoded and routed properly rather than causing a 500.
        r = client.get("/cities/Den Haag")
        assert r.status_code == 404
''')
run_pytest("test_exercise1.py", "-v")
```

The last one matters more than it looks. Spaces are illegal in raw URLs; the client encodes it as
`Den%20Haag` and FastAPI decodes it. If you ever build a URL by gluing strings together you'll break
this, which is why Module 2's `fetch.py` passes `params={...}` instead.
</details>

### Exercise 2 — A fixture that guarantees cleanup (⭐⭐)

Write a fixture `empty_client` that overrides `get_conn` with a connection returning **no** rows, and
prove with a second test that the override did **not** leak — i.e. the normal `client` fixture still
sees three cities afterwards.

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
write("test_exercise2.py", '''
    import pytest
    from fastapi.testclient import TestClient
    from app_under_test import app, get_conn

    class EmptyConnection:
        def query(self, sql):
            return []

    @pytest.fixture
    def empty_client():
        def override():
            yield EmptyConnection()
        app.dependency_overrides[get_conn] = override
        yield TestClient(app)
        app.dependency_overrides.clear()          # <- the guarantee

    def test_a_empty(empty_client):
        assert empty_client.get("/cities").json() == []

    def test_b_not_leaked(client):
        # Runs AFTER test_a. If the override leaked, this sees [] and fails.
        assert len(client.get("/cities").json()) == 3
''')
run_pytest("test_exercise2.py", "-v")
```

Now delete the `app.dependency_overrides.clear()` line and re-run. `test_b_not_leaked` fails —
and note it fails *only because of the order tests ran in*. That class of bug is miserable to
diagnose in a large suite, which is why the teardown is not optional.
</details>

### Exercise 3 — Assert on what the query received (⭐⭐⭐)

Make the fake record every SQL string it is asked for, then write a test proving `GET /cities` really
does query the database (rather than returning something hard-coded). Then extend it to assert the
`limit` was applied.

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
write("test_exercise3.py", '''
    import pytest
    from fastapi.testclient import TestClient
    from app_under_test import app, get_conn

    ROWS = [{"city": "Utrecht"}, {"city": "Amsterdam"}, {"city": "Rotterdam"}]

    class RecordingConnection:
        def __init__(self):
            self.queries = []
        def query(self, sql):
            self.queries.append(sql)
            return ROWS

    @pytest.fixture
    def recorder():
        rec = RecordingConnection()
        def override():
            yield rec
        app.dependency_overrides[get_conn] = override
        yield rec, TestClient(app)
        app.dependency_overrides.clear()

    def test_it_actually_queries(recorder):
        rec, client = recorder
        client.get("/cities")
        assert len(rec.queries) == 1
        assert "FROM cities" in rec.queries[0]

    def test_limit_applied(recorder):
        rec, client = recorder
        assert len(client.get("/cities", params={"limit": 2}).json()) == 2
''')
run_pytest("test_exercise3.py", "-v")
```

A recording fake is the middle ground between a dumb stub and a real database: you can assert on the
**interaction**, not just the result. Use it sparingly — asserting on exact SQL strings couples your
tests to your implementation, and then a harmless rewrite turns the suite red. Assert on *what was
asked for*, not *how it was phrased*.
</details>

---

## 🧹 Clean up

`sandbox/` is git-ignored. Run this to remove it, or leave it and re-run the notebook — every cell
overwrites its own files.

In [ ]:
import shutil

sys.path = [p for p in sys.path if p != str(SANDBOX.resolve())]
shutil.rmtree(SANDBOX, ignore_errors=True)
print("sandbox removed")

---

## ✅ Before you move on

- Why does `TestClient` need no server and no port?
- What single line swaps a real database for a fake, and why does the handler not notice?
- When is exact-equality assertion right, and when is it a trap?
- What are the two tests people usually skip for an endpoint?
- What does `12 passed, 4 skipped` actually prove about your SQL?

**That's Week A.** You can build and test an API.
**Week B ships it** — starting with [Day 6 theory](../day6-async/11_theory_async_basics.md).